In [ ]:
if 'google.colab' in str(get_ipython()):
  !pip install git+https://github.com/FullControlXYZ/fullcontrol --quiet
import lab.fullcontrol.infinaxis as fci
from math import sin, cos, tau


In [ ]:
EW = 0.6
EH = 0.3

print_settings = {'extrusion_width': EW,'extrusion_height': EH}
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
Point = fci.configure_point(head_chain, bed_chain)

gcode_controls = fci.GcodeControls(
    head_chain = head_chain,
    bed_chain = bed_chain,
    initialization_data=print_settings 
    )

#### Five-axis spiral path

This path uses one spiral turn per layer. `B` tilt increases by 1 degree per layer for 90 layers. The layer-to-layer spacing is kept constant by splitting the same step length into radial and vertical components with sine/cosine.


In [ ]:
density = 360
layers = 90
layer_gap = EH
r = 10
z = EH / 2
steps = []

for layer in range(layers):
    tilt = layer
    dr = layer_gap * sin(tilt / 360 * tau)
    dz = layer_gap * cos(tilt / 360 * tau)
    for j in range(density):
        t = j / density
        angle = 360 * (layer + t)
        r_now = r + dr * t
        z_now = z + dz * t
        steps.append(Point(x=r_now * sin(angle / 360 * tau), y=r_now * cos(angle / 360 * tau), z=z_now, b=tilt, c=angle))
    r += dr
    z += dz

for step in steps:
    step.color = [abs(step.b) / 90, 0, 1 - abs(step.b) / 90]

fci.transform(steps, 'plot', fci.PlotControls(color_type='manual', style='tube', zoom=0.75), show_tips=False)

gcode = fci.transform(steps, 'gcode', gcode_controls)
print('first ten gcode lines:\n' + '\n'.join(gcode.split('\n')[:10]))
print('')
print('final ten gcode lines:\n' + '\n'.join(gcode.split('\n')[-10:]))
